# Experiment 7
## Model Performance Evaluation on Validation Set
**Aim:** Evaluate a trained model's accuracy and loss on a separate validation set.

### Step 1 – Import Libraries

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

### Step 2 – Load and Split Dataset into Train / Validation

In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
full_data = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_data, val_data = random_split(full_data, [50000, 10000])
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_data,   batch_size=64, shuffle=False)
print(f'Train: {len(train_data)} | Validation: {len(val_data)}')

### Step 3 – Define and Train a Simple Model

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))
    def forward(self, x): return self.fc(x)

model     = Net()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

### Step 4 – Training Loop with Validation After Each Epoch

In [ ]:
train_losses, val_losses, val_accs = [], [], []
for epoch in range(5):
    # --- Training ---
    model.train()
    t_loss = 0
    for X, y in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward(); optimizer.step()
        t_loss += loss.item()

    # --- Validation ---
    model.eval()
    v_loss = correct = total = 0
    with torch.no_grad():
        for X, y in val_loader:
            out = model(X)
            v_loss  += criterion(out, y).item()
            correct += (out.argmax(1) == y).sum().item()
            total   += y.size(0)
    train_losses.append(t_loss / len(train_loader))
    val_losses.append(v_loss  / len(val_loader))
    val_accs.append(100 * correct / total)
    print(f'Epoch {epoch+1} | Train Loss: {train_losses[-1]:.4f} | Val Loss: {val_losses[-1]:.4f} | Val Acc: {val_accs[-1]:.2f}%')

### Step 5 – Plot Training vs Validation Loss

In [ ]:
import matplotlib.pyplot as plt
epochs = range(1, 6)
plt.figure(figsize=(12, 4))
plt.subplot(1,2,1)
plt.plot(epochs, train_losses, 'b-o', label='Train Loss')
plt.plot(epochs, val_losses,   'r-o', label='Val Loss')
plt.title('Loss'); plt.xlabel('Epoch'); plt.legend(); plt.grid(True)
plt.subplot(1,2,2)
plt.plot(epochs, val_accs, 'g-o', label='Val Accuracy')
plt.title('Validation Accuracy'); plt.xlabel('Epoch'); plt.ylabel('%'); plt.legend(); plt.grid(True)
plt.tight_layout(); plt.show()

### Result
The model was evaluated on a validation set after every epoch. Loss decreased and accuracy improved over training, showing successful learning.